In [ ]:
# If running in Google Colab, before beginning, run this cell to install optimization packages
!pip install pyomo
!apt-get install -y -qq glpk-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 1.8 MB/s eta 0:00:00
Selecting previously unselected package libsuitesparseconfig5:amd64.
(Reading database ... 123630 files and directories currently installed.)
Preparing to unpack .../libsuitesparseconfig5_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libsuitesparseconfig5:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libamd2:amd64.
Preparing to unpack .../libamd2_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libamd2:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libcolamd2:amd64.
Preparing to unpack .../libcolamd2_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libcolamd2:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libglpk40:amd64.
Preparing to unpack .../libglpk40_5.0-1_amd64.deb ...
Unpacking libglpk40:amd64 (5.0-1) ...
Selecting previously unselected package

# Advanced Linearization

In the last lesson, we took an advertising model optimizing the number of people reached by ads, where number of customers reached by online and TV ads were piecewise linear functions of online and TV ads per day, and converted this to a linear model. Let's look at how we would go about implementing and solving this linearized model in Pyomo.

The linearized model was as follows:



*Objective :*

$ \text{Maximize } x + y \quad \text{(maximize people reached)} $

*Subject to:*

$ 1000x + 10000y \leq 150000 \quad \text{(budget constraint)} $

$ x \leq 900w $

$ x \leq 3000 + 600w $

$ x \leq 9000 + 300w $

$ y \leq 10000t $

$ y \leq 25000 + 5000t $

$ y \leq 55000 + 2000t $

$ w \leq 30 $

$ t \leq 15 $

$ w, t \geq 0 \quad \text{(cannot place negative number of ads per day in each medium).} $


The first three constraints for $x$ ensure that at optimality:
  $ x = \min\{900w, 3000 + 600w, 9000 + 300w\}. $ Similarly, the three constraints for $y$ ensure that at optimality:
  $ y = \min\{10000t, 25000 + 5000t, 55000 + 2000t\}. $ Lastly, $w$ and $t$ have upper bounds, and both must be non-negative.

Now that this model has been linearized, implementing and solving it in Pyomo is straightforward, it is simply a larger model that we have seen before. See if you can match each of the mathemetical constraints with their coded form below, then run the cell to solve the model:

In [ ]:
# Import Pyomo library
from pyomo.environ import *

# Create a Pyomo model
model = ConcreteModel()

# Define decision variables
model.x = Var(domain=NonNegativeReals)  # Number of people reached via online ads
model.y = Var(domain=NonNegativeReals)  # Number of people reached via TV ads
model.w = Var(domain=NonNegativeReals)  # Number of ads online per day
model.t = Var(domain=NonNegativeReals)  # Number of ads on TV per day

# Define the objective function: Maximize people reached
model.objective = Objective(
    expr=model.x + model.y,
    sense=maximize
)

# Define the budget constraint
model.budget_constraint = Constraint(
    expr=1000 * model.x + 10000 * model.y <= 150000
)

# Define the constraints for x
model.x_constraint1 = Constraint(expr=model.x <= 900 * model.w)
model.x_constraint2 = Constraint(expr=model.x <= 3000 + 600 * model.w)
model.x_constraint3 = Constraint(expr=model.x <= 9000 + 300 * model.w)

# Define the constraints for y
model.y_constraint1 = Constraint(expr=model.y <= 10000 * model.t)
model.y_constraint2 = Constraint(expr=model.y <= 25000 + 5000 * model.t)
model.y_constraint3 = Constraint(expr=model.y <= 55000 + 2000 * model.t)

# Define the upper limits for w and t
model.w_limit = Constraint(expr=model.w <= 30)
model.t_limit = Constraint(expr=model.t <= 15)

# Solve the model
solver = SolverFactory('glpk')  # Using GLPK solver
result = solver.solve(model)

# Display the results
print("Solver Status:", result.solver.status)
print("Termination Condition:", result.solver.termination_condition)
print("\nOptimal Solution:")
print(f"x = {model.x.value:.2f} (people reached via online ads")
print(f"y = {model.y.value:.2f} (people reached via TV adds")
print(f"w = {model.w.value:.2f} (ads placed online)")
print(f"t = {model.t.value:.2f} (ads placed on TV)")
print(f"Optimal Objective Value (Total People Reached) = {model.objective():.2f}")


Solver Status: ok
Termination Condition: optimal

Optimal Solution:
x = 150.00 (people reached via online ads
y = 0.00 (people reached via TV adds
w = 0.17 (ads placed online)
t = 0.00 (ads placed on TV)
Optimal Objective Value (Total People Reached) = 150.00


It seems that in this case, we should dedicate all of our effors to online advertising!